In [2]:
import os
import sys

# Force PySpark driver and workers to use the exact same Python executable as Jupyter
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

import pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, date_trunc, sum as spark_sum, avg as spark_avg, 
    to_timestamp
)

# 1. Restart SparkSession with the matching Python executable
if 'spark' in locals() or 'spark' in globals():
    spark.stop()

spark = SparkSession.builder \
    .appName("EV_Grid_Load_Preprocessing") \
    .config("spark.driver.memory", "4g") \
    .config("spark.pyspark.python", sys.executable) \
    .config("spark.pyspark.driver.python", sys.executable) \
    .getOrCreate()

# 2. Base Directory Paths
base_dir = r"C:\Data_from_shahana_onedrive\CIMT\FinalCapstoneProject\Project_files\DataSets\evwatts.public\evwatts.public"

session_uri = Path(os.path.join(base_dir, "evwatts.public.session.csv")).as_uri()
weather_path = os.path.join(base_dir, "weather_data.csv")

# 3. Load EV Session Data
df_session_raw = spark.read.csv(session_uri, header=True, inferSchema=True)

# 4. Clean & Truncate EV Session Timestamps to Hourly Grain
df_session_clean = df_session_raw \
    .filter(col("energy_kwh").isNotNull() & (col("energy_kwh") > 0)) \
    .withColumn("timestamp", date_trunc("hour", to_timestamp(col("start_datetime")))) \
    .groupBy("timestamp") \
    .agg(
        spark_sum("energy_kwh").alias("total_kwh_demand"),
        spark_avg("energy_kwh").alias("avg_session_kwh")
    )

# 5. Read Weather File & Dynamically Skip Metadata Header
with open(weather_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

header_idx = next(i for i, line in enumerate(lines) if line.startswith("time"))
pdf_weather = pd.read_csv(weather_path, skiprows=header_idx)
df_weather_raw = spark.createDataFrame(pdf_weather)

# 6. Clean Weather Timestamps and Rename Unit-Suffixed Columns
df_weather_clean = df_weather_raw \
    .withColumn("timestamp", date_trunc("hour", to_timestamp(col("time")))) \
    .select(
        "timestamp", 
        col("`temperature_2m (°C)`").cast("double").alias("temperature_2m"), 
        col("`relative_humidity_2m (%)`").cast("double").alias("relative_humidity_2m"), 
        col("`precipitation (mm)`").cast("double").alias("precipitation"), 
        col("`direct_radiation (W/m²)`").cast("double").alias("direct_radiation")
    )

# 7. Inner Join Hourly Grid Load with Clean Weather Metrics
df_hourly_grid = df_session_clean \
    .join(df_weather_clean, on="timestamp", how="inner") \
    .orderBy("timestamp")

print("--- Cleaned & Joined Hourly Dataset Preview ---")
df_hourly_grid.show(10, truncate=False)

--- Cleaned & Joined Hourly Dataset Preview ---
+-------------------+------------------+------------------+--------------+--------------------+-------------+----------------+
|timestamp          |total_kwh_demand  |avg_session_kwh   |temperature_2m|relative_humidity_2m|precipitation|direct_radiation|
+-------------------+------------------+------------------+--------------+--------------------+-------------+----------------+
|2019-06-28 13:00:00|6.211             |6.211             |10.3          |96.0                |0.0          |2.0             |
|2019-07-01 08:00:00|6.316             |6.316             |17.7          |68.0                |0.0          |0.0             |
|2019-07-01 13:00:00|3.886             |3.886             |15.3          |80.0                |0.0          |1.0             |
|2019-07-01 15:00:00|9.927999999999999 |9.927999999999999 |17.2          |69.0                |0.0          |118.0           |
|2019-07-02 09:00:00|6.2589999999999995|6.2589999999999995|13.8

# 🛠️ Complete Phase 2: Time Sequence, Feature Engineering & Parquet Export
Execute the following cell in your Jupyter Notebook to:

Generate a continuous hourly timeline from 2019-06-25 to 2022-12-31.

Fill missing demand hours with 0.0 kWh.

Engineered Features:

Calendar Features: hour, day_of_week, month, is_weekend

Temporal Lags: 1-hour lag (kwh_lag_1h), 24-hour lag (kwh_lag_24h)

Rolling Averages: 24-hour rolling average load (kwh_rolling_avg_24h)

Save to Parquet: Write the final feature matrix into compressed Parquet files for fast ML training in Phase 3.

In [4]:
import os
from pyspark.sql.functions import (
    col, sequence, expr, hour, dayofweek, month, 
    when, coalesce, lit, lag, avg as spark_avg
)
from pyspark.sql.window import Window

print("--- Step-by-Step Row Count Check ---")

# 1. Check date bounds of your session and weather data
session_range = df_session_clean.selectExpr("min(timestamp)", "max(timestamp)").collect()[0]
weather_range = df_weather_clean.selectExpr("min(timestamp)", "max(timestamp)").collect()[0]

print(f"EV Sessions Date Range: {session_range[0]} to {session_range[1]} | Count: {df_session_clean.count()}")
print(f"Weather Data Date Range: {weather_range[0]} to {weather_range[1]} | Count: {df_weather_clean.count()}")

# Use dynamic min/max timestamps based on your actual data bounds
min_ts = session_range[0] if session_range[0] else "2019-06-25 00:00:00"
max_ts = session_range[1] if session_range[1] else "2022-12-31 23:00:00"

# 2. Continuous Hourly Grid
time_range_df = spark.sql(f"""
    SELECT explode(sequence(
        to_timestamp('{min_ts}'), 
        to_timestamp('{max_ts}'), 
        interval 1 hour
    )) AS timestamp
""")
print(f"Generated Timeline Row Count: {time_range_df.count()}")

# 3. Left Join Continuous Grid
df_continuous_grid = time_range_df \
    .join(df_weather_clean, on="timestamp", how="left") \
    .join(df_session_clean, on="timestamp", how="left") \
    .select(
        col("timestamp"),
        coalesce(col("total_kwh_demand"), lit(0.0)).alias("total_kwh_demand"),
        coalesce(col("avg_session_kwh"), lit(0.0)).alias("avg_session_kwh"),
        coalesce(col("temperature_2m"), lit(0.0)).alias("temperature_2m"),
        coalesce(col("relative_humidity_2m"), lit(0.0)).alias("relative_humidity_2m"),
        coalesce(col("precipitation"), lit(0.0)).alias("precipitation"),
        coalesce(col("direct_radiation"), lit(0.0)).alias("direct_radiation")
    )
print(f"Continuous Grid Joined Row Count: {df_continuous_grid.count()}")

# 4. Feature Engineering (Fill initial NA lags with 0.0 instead of dropping rows)
window_spec = Window.orderBy("timestamp")
window_24h = Window.orderBy("timestamp").rowsBetween(-23, 0)

df_engineered = df_continuous_grid \
    .withColumn("hour", hour(col("timestamp"))) \
    .withColumn("day_of_week", dayofweek(col("timestamp"))) \
    .withColumn("month", month(col("timestamp"))) \
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("kwh_lag_1h", coalesce(lag(col("total_kwh_demand"), 1).over(window_spec), lit(0.0))) \
    .withColumn("kwh_lag_24h", coalesce(lag(col("total_kwh_demand"), 24).over(window_spec), lit(0.0))) \
    .withColumn("kwh_rolling_avg_24h", coalesce(spark_avg(col("total_kwh_demand")).over(window_24h), lit(0.0)))

print(f"Final Engineered Matrix Row Count: {df_engineered.count()}")

# 5. Clean Column Names & Export Safely
# Ensure all column names are alphanumeric/underscores only
clean_cols = [c.replace(" ", "_").replace("(", "").replace(")", "") for c in df_engineered.columns]
df_engineered_clean = df_engineered.toDF(*clean_cols)

parquet_output_path = os.path.join(base_dir, "ev_weather_processed.parquet")

try:
    # Attempt direct single-file Parquet write via Spark
    df_engineered_clean.coalesce(1).write.mode("overwrite").parquet(parquet_output_path)
    print("✅ PySpark Parquet export successful!")
except Exception as e:
    print(f"⚠️ PySpark native write encountered a file lock, falling back to Pandas export...\nError: {e}")
    # Fallback: Convert to Pandas and export via PyArrow/Fastparquet
    pdf_final = df_engineered_clean.toPandas()
    pdf_final.to_parquet(parquet_output_path, index=False)
    print("✅ Pandas Parquet export successful!")

# Verify written files
if os.path.exists(parquet_output_path):
    if os.path.isdir(parquet_output_path):
        files = os.listdir(parquet_output_path)
        print(f"Directory Contents: {files}")
    else:
        print(f"Single Parquet file created: {os.path.getsize(parquet_output_path) / 1024:.2f} KB")

--- Step-by-Step Row Count Check ---
EV Sessions Date Range: 2019-06-28 13:00:00 to 2022-12-31 23:00:00 | Count: 28660
Weather Data Date Range: 2019-06-25 00:00:00 to 2022-12-31 23:00:00 | Count: 30864
Generated Timeline Row Count: 30780
Continuous Grid Joined Row Count: 30783
Final Engineered Matrix Row Count: 30783
⚠️ PySpark native write encountered a file lock, falling back to Pandas export...
Error: An error occurred while calling o418.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSy

PermissionError: [WinError 5] Failed to open local file 'C:/Data_from_shahana_onedrive/CIMT/FinalCapstoneProject/Project_files/DataSets/evwatts.public/evwatts.public/ev_weather_processed.parquet'. Detail: [Windows error 5] Access is denied.


In [6]:
# Clean column names (replace spaces and parentheses)
clean_cols = [c.replace(" ", "_").replace("(", "").replace(")", "") for c in df_engineered.columns]
df_engineered_clean = df_engineered.toDF(*clean_cols)

# Output File Path
parquet_output_file = os.path.join(base_dir, "ev_weather_processed.parquet")

print("Exporting feature matrix to Parquet via Pandas...")
# Convert to Pandas DataFrame and export directly
pdf_final = df_engineered_clean.toPandas()
pdf_final.to_parquet(parquet_output_file, index=False)

# Verification
file_size_mb = os.path.getsize(parquet_output_file) / (1024 * 1024)
print(f"✅ Success! Feature matrix saved cleanly to:\n{parquet_output_file}")
print(f"📊 Dataset File Size: {file_size_mb:.2f} MB ({pdf_final.shape[0]} rows, {pdf_final.shape[1]} columns)")

Exporting feature matrix to Parquet via Pandas...
✅ Success! Feature matrix saved cleanly to:
C:\Data_from_shahana_onedrive\CIMT\FinalCapstoneProject\Project_files\DataSets\evwatts.public\evwatts.public\ev_weather_processed.parquet
📊 Dataset File Size: 1.72 MB (30783 rows, 14 columns)
